# Sistem Case-Based Reasoning (CBR) - Analisis Putusan Hukum

Notebook ini berisi implementasi sistem **Case-Based Reasoning (CBR)** untuk analisis dokumen putusan perdata agama perceraian di Indonesia. Alur sistem mencakup pemrosesan teks dokumen PDF, ekstraksi fitur kasus, pencarian kemiripan (*similarity retrieval*), hingga penyimpanan kasus baru (*retain*).

--- 
## 1. Persiapan dan Import Library
Langkah awal adalah mengimpor pustaka (*library*) yang diperlukan untuk pengolahan berkas, pemrosesan bahasa alami (NLP), dan pemodelan CBR.

In [ ]:
import os
import re
import json
import pandas as pd
import numpy as np
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Pustaka khusus bahasa Indonesia
# Jika Sastrawi belum terinstall, silahkan jalankan pip install Sastrawi di terminal
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

print("Library berhasil di-import!")

--- 
## 2. Pra-pemrosesan Data (Pre-processing)
Pada tahap ini kita akan membaca berkas putusan hukum (format PDF) dari direktori `data/raw/` dan mengekstrak teksnya, lalu melakukan pembersihan teks (case folding, filtering, stemming).

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Mengekstrak seluruh teks dari berkas PDF."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

def clean_text(text):
    """Melakukan case folding dan membersihkan tanda baca / angka."""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Hapus karakter non-huruf
    text = re.sub(r'\s+', ' ', text).strip() # Hapus spasi berlebih
    return text

def stem_indonesian(text):
    """Melakukan stemming bahasa Indonesia menggunakan Sastrawi."""
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()
    return stemmer.stem(text)

# Contoh penggunaan (silahkan sesuaikan nama berkas jika sudah memindahkan PDF ke data/raw/):
# text_raw = extract_text_from_pdf("data/raw/putusan_xxx.pdf")
# text_clean = clean_text(text_raw)
# text_stemmed = stem_indonesian(text_clean)

--- 
## 3. Ekstraksi Fitur & Pembentukan Basis Kasus (*Case Base*)
Kasus hukum direpresentasikan dalam bentuk fitur terstruktur (seperti alasan perceraian, tuntutan nafkah, hak asuh anak) maupun representasi teks (TF-IDF Vector).

In [ ]:
# Struktur data representasi Kasus (Case Base)
case_base = [
    {
        "id_kasus": "K-01",
        "nomor_putusan": "Putusan A",
        "alasan_utama": "Perselisihan terus menerus akibat masalah ekonomi",
        "hak_asuh_anak": "Ibu",
        "tuntutan_nafkah": "Dikabulkan sebagian",
        "teks_lengkap": "penggugat mengajukan gugatan cerai karena tergugat tidak memberi nafkah dan sering bertengkar..."
    },
    {
        "id_kasus": "K-02",
        "nomor_putusan": "Putusan B",
        "alasan_utama": "KDRT (Kekerasan Dalam Rumah Tangga)",
        "hak_asuh_anak": "Ibu",
        "tuntutan_nafkah": "Dikabulkan sepenuhnya",
        "teks_lengkap": "tergugat melakukan kekerasan fisik terhadap penggugat sejak tahun kedua pernikahan..."
    }
]

df_cases = pd.DataFrame(case_base)
df_cases

--- 
## 4. Mesin Case-Based Reasoning (CBR Engine)
Bagian ini mengimplementasikan siklus **Retrieve**, **Reuse**, **Revise**, dan **Retain**.

### A. Retrieve (Menemukan Kembali)
Mencari kasus terdekat dengan menghitung nilai kemiripan (*Similarity*) menggunakan Cosine Similarity pada representasi TF-IDF teks putusan.

In [ ]:
def retrieve_similar_cases(query_text, df_cases, top_n=1):
    """Mencari kasus terdekat berdasarkan kecocokan teks menggunakan TF-IDF & Cosine Similarity."""
    # Gabungkan teks query dengan teks kasus yang ada
    corpus = list(df_cases['teks_lengkap']) + [query_text]
    
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus)
    
    # Hitung cosine similarity antara query (indeks terakhir) dengan seluruh kasus (indeks 0 s.d N-1)
    similarity_scores = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])[0]
    
    # Urutkan berdasarkan kemiripan tertinggi
    results = []
    for idx, score in enumerate(similarity_scores):
        results.append({
            "id_kasus": df_cases.iloc[idx]['id_kasus'],
            "nomor_putusan": df_cases.iloc[idx]['nomor_putusan'],
            "alasan_utama": df_cases.iloc[idx]['alasan_utama'],
            "hak_asuh_anak": df_cases.iloc[idx]['hak_asuh_anak'],
            "similarity": score
        })
    
    results_sorted = sorted(results, key=lambda x: x['similarity'], reverse=True)
    return results_sorted[:top_n]

# Uji coba pencarian dengan kasus baru (query)
kasus_baru = "suami melakukan pemukulan kdrt dan tidak memberikan nafkah lahir batin"
kasus_terdekat = retrieve_similar_cases(kasus_baru, df_cases, top_n=1)
print("Kasus Paling Mirip yang Ditemukan:")
print(json.dumps(kasus_terdekat, indent=4))

### B. Reuse & Revise (Menggunakan & Merevisi Solusi)
Menyalin solusi dari kasus serupa dan menyesuaikannya jika ada perbedaan detail signifikan pada kasus baru.

In [ ]:
def reuse_and_revise(retrieved_case, new_case_details):
    """Merekomendasikan solusi berdasarkan kasus terdekat dan merevisinya sesuai kebutuhan."""
    rekomendasi_solusi = {
        "hak_asuh_anak": retrieved_case['hak_asuh_anak'],
        "status_solusi": "Diadopsi dari " + retrieved_case['id_kasus']
    }
    
    # Aturan revisi sederhana:
    # Jika dalam detail kasus baru anak masih di bawah umur (<12 tahun) dan ibu terbukti mampu secara moral,
    # hak asuh tetap direkomendasikan pada ibu.
    if new_case_details.get('anak_dibawah_umur') == True:
        rekomendasi_solusi['hak_asuh_anak'] = "Ibu (Revisi: Anak di bawah umur wajib dirawat Ibu)"
        rekomendasi_solusi['status_solusi'] = "Direvisi sesuai undang-undang perlindungan anak"
        
    return rekomendasi_solusi

detail_kasus_baru = {"anak_dibawah_umur": True}
solusi_diusulkan = reuse_and_revise(kasus_terdekat[0], detail_kasus_baru)
print("Solusi yang Diusulkan:", solusi_diusulkan)

### C. Retain (Menyimpan Kasus Baru)
Jika hasil revisi kasus baru ini valid dan telah terverifikasi, kita akan menyimpannya ke dalam Case Base agar sistem bertambah cerdas seiring waktu.

In [ ]:
def retain_new_case(df_cases, id_baru, nomor_baru, alasan, hak_asuh, teks):
    """Menambahkan kasus baru yang telah sukses diselesaikan ke basis kasus."""
    new_row = {
        "id_kasus": id_baru,
        "nomor_putusan": nomor_baru,
        "alasan_utama": alasan,
        "hak_asuh_anak": hak_asuh,
        "tuntutan_nafkah": "N/A",
        "teks_lengkap": teks
    }
    df_updated = pd.concat([df_cases, pd.DataFrame([new_row])], ignore_index=True)
    return df_updated

df_cases = retain_new_case(
    df_cases, 
    id_baru="K-03", 
    nomor_baru="Putusan C", 
    alasan="KDRT dan Anak di bawah umur", 
    hak_asuh=solusi_diusulkan['hak_asuh_anak'],
    teks=kasus_baru
)
print("Case Base Terkini setelah Retain:")
df_cases

--- 
## 5. Evaluasi Sistem (Evaluation)
Menguji performa penemuan kembali (*retrieval*) dengan metrik kecocokan atau membandingkan akurasi klasifikasi hak asuh anak / alasan utama perceraian antara rekomendasi sistem dengan vonis hakim yang sebenarnya.

In [ ]:
# Simpan basis kasus terupdate ke folder data/processed/ jika diperlukan
# df_cases.to_csv("data/processed/case_base.csv", index=False)
# print("Kasus berhasil diekspor ke data/processed/case_base.csv")